# Day 31: Implement SQL RAG

Welcome to Day 31! Today we are tackling **SQL RAG** (Retrieval-Augmented Generation on Structured Data). We will use LangChain to translate natural language queries into executable SQL against a SQLite database.

## Core Theory (Just-in-Time)

### The "Why"
Most enterprise data doesn't live in PDFs or unstructured text; it lives in relational databases. Traditional RAG (chunking text and using vector search) fails miserably on structured, tabular data. You can't run a `SUM()`, `GROUP BY`, or `JOIN` using a vector similarity search. To unlock insights from relational databases, we need the LLM to act as a translator—turning a user's natural language question into a syntactically correct SQL query.

### The "How"
The architecture for SQL RAG typically follows these steps:
1.  **Schema Retrieval:** The LLM is provided with the database dialect and the schema (table names, column names, data types, and perhaps sample rows) of the relevant tables.
2.  **Query Generation:** The LLM generates a SQL query based on the user's question and the retrieved schema.
3.  **Query Execution:** The generated SQL is executed safely against the database.
4.  **Answer Synthesis:** The results of the query are fed back into the LLM to generate a natural language response summarizing the findings.

We will use LangChain's standard tools (specifically `create_sql_query_chain`) to manage this workflow.

## AI Security & Common Pitfalls in Production
1.  **Security Risks (SQL Injection / Destructive Queries):** Never give the LLM a connection with write/drop permissions. **Always use read-only credentials.** If the LLM hallucinates a `DROP TABLE`, it should fail at the database permission level.
2.  **PII Data Protection:** Do not allow the LLM to indiscriminately query and return sensitive Personally Identifiable Information (PII). Implement column-level security or explicit prompt instructions to redact PII (e.g., masking emails or phone numbers).
3.  **Fallback Mechanisms:** If the LLM generates syntactically invalid SQL, the execution will crash. A robust system should catch the exception and optionally retry the generation or gracefully return a fallback message to the user.
4.  **Schema Complexity:** Feeding an entire enterprise database schema into an LLM context window will fail. You must filter or retrieve only the relevant table schemas *before* generation.

## Reference Links
- [LangChain SQL Database Tools](https://python.langchain.com/docs/use_cases/sql/)
- [SQLAlchemy Documentation](https://docs.sqlalchemy.org/en/20/)
- [OWASP Top Ten (SQL Injection)](https://owasp.org/www-community/attacks/SQL_Injection)


In [1]:
# Run this in your terminal if you haven't already:
# uv pip install langchain langchain-openai langchain-community sqlalchemy

## 1. Database Setup

First, let's create a sample SQLite database using `sqlite3` and `sqlalchemy` to act as our data source.

In [2]:
import sqlite3
from typing import List, Tuple

def setup_sample_db(db_path: str = "ecommerce.db") -> None:
    """Creates a sample SQLite database with users and orders tables."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Create tables
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            user_id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            signup_date DATE
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id INTEGER PRIMARY KEY,
            user_id INTEGER,
            product_name TEXT NOT NULL,
            amount REAL NOT NULL,
            order_date DATE,
            FOREIGN KEY(user_id) REFERENCES users(user_id)
        )
    ''')

    # Insert sample data
    cursor.execute("DELETE FROM users")
    cursor.execute("DELETE FROM orders")
    
    users: List[Tuple[int, str, str, str]] = [
        (1, 'Alice Smith', 'alice@example.com', '2023-01-15'),
        (2, 'Bob Johnson', 'bob@example.com', '2023-02-20'),
        (3, 'Charlie Brown', 'charlie@example.com', '2023-03-10')
    ]
    cursor.executemany("INSERT INTO users VALUES (?, ?, ?, ?)", users)

    orders: List[Tuple[int, int, str, float, str]] = [
        (101, 1, 'Laptop', 1200.50, '2023-04-01'),
        (102, 1, 'Mouse', 25.00, '2023-04-02'),
        (103, 2, 'Monitor', 300.00, '2023-04-10'),
        (104, 3, 'Keyboard', 75.00, '2023-04-15'),
        (105, 1, 'Headphones', 150.00, '2023-04-20')
    ]
    cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)

    conn.commit()
    conn.close()
    print(f"Sample database initialized at {db_path}")

setup_sample_db()

Sample database initialized at ecommerce.db


## 2. Implementing SQL RAG with LangChain

Now we will connect LangChain to our database and build a chain that translates a natural language question into a SQL query.

### Basic Implementation
Isolate the core concept: generating a SQL query from natural language with minimal boilerplate.

In [3]:
import os
from langchain_community.utilities.sql_database import SQLDatabase
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI

# 1. Connect to the database using LangChain's wrapper
db = SQLDatabase.from_uri("sqlite:///ecommerce.db")

# 2. Initialize the LLM
# We use temperature=0 for deterministic SQL generation
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)\n
# 3. Create the SQL Query Chain
generate_query_chain = create_sql_query_chain(llm, db)

# 4. Generate the query
question = "How many users are in the database?"
response = generate_query_chain.invoke({"question": question})
print(f"Question: {question}\nGenerated SQL: {response}")

Question: How many users are in the database?
Generated SQL: SELECT 1


### Medium Implementation
Emphasize clean OOP and state management. We encapsulate the logic within a class, properly initialize components, and execute the generated query.

In [4]:
from typing import Optional
from langchain_community.utilities.sql_database import SQLDatabase
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI

class SimpleSQLRAG:
    """A simple class-based SQL RAG implementation."""
    
    def __init__(self, db_uri: str, model_name: str = "gpt-3.5-turbo"):
        self.db = SQLDatabase.from_uri(db_uri)
        self.llm = ChatOpenAI(model=model_name, temperature=0)\n        self.query_chain = create_sql_query_chain(self.llm, self.db)
        
    def _extract_sql(self, text: str) -> str:
        """Cleans up the SQL string returned by the LLM by removing markdown formatting."""
        return text.replace("```sql", "").replace("```", "").strip()

    def query(self, question: str) -> Optional[str]:
        """Generates and executes a SQL query based on the user's question."""
        try:
            # 1. Generate Query
            generated_sql_raw = self.query_chain.invoke({"question": question})
            clean_sql = self._extract_sql(generated_sql_raw)
            print(f"Generated SQL: \n{clean_sql}\n")
            
            # 2. Execute Query
            # WARNING: This executes the query directly. Ensure read-only privileges in production!
            result = self.db.run(clean_sql)
            return result
        except Exception as e:
            print(f"Error executing query: {e}")
            return None

# Test the medium implementation
sql_rag = SimpleSQLRAG("sqlite:///ecommerce.db")
result = sql_rag.query("What is the total amount spent by Alice Smith?")
print(f"Execution Result: {result}")

Generated SQL: 
SELECT 1

Execution Result: [(1,)]


### Advanced Implementation
Production-grade implementation with strict type hinting, docstrings, error handling, exact imports, AI security practices (fallbacks and PII awareness), and answer synthesis.

In [5]:
from typing import Optional, Dict, Any
import logging
from pydantic import BaseModel, Field

from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class QueryResult(BaseModel):
    """Standardized response format for SQL RAG queries."""
    question: str
    sql_query: Optional[str] = Field(default=None, description="The generated SQL query.")
    result: Optional[str] = Field(default=None, description="The raw database result.")
    synthesized_answer: str = Field(..., description="The final natural language answer.")
    error: Optional[str] = Field(default=None, description="Any error encountered during execution.")

class ProductionSQLRAGAgent:
    """
    Production-grade SQL RAG Agent.
    Demonstrates clean OOP, state management, and incorporates security best practices:
    - Read-only DB connection (conceptually demonstrated via comments; enforce at DB level).
    - Robust error handling and fallback mechanisms.
    - Prompts designed to mitigate PII leakage.
    """

    def __init__(self, db_uri: str, model_name: str = "gpt-3.5-turbo"):
        """
        Initializes the ProductionSQLRAGAgent.
        
        Args:
            db_uri: The SQLAlchemy database URI.
            model_name: The OpenAI model to use.
        """
        try:
            # In production, ensure db_uri uses credentials restricted to read-only access
            self.db = SQLDatabase.from_uri(db_uri)
            self.llm = ChatOpenAI(model=model_name, temperature=0)\n            
            # Setup tools and chains
            self.execute_query_tool = QuerySQLDataBaseTool(db=self.db)
            self.write_query_chain = create_sql_query_chain(self.llm, self.db) | self._extract_sql
            
            # Setup Answer Synthesis Prompt (incorporating PII security instructions)
            self.answer_prompt = PromptTemplate.from_template(
                """Given the following user question, corresponding SQL query, and SQL result, answer the user question.
                
                SECURITY INSTRUCTION: If the result contains sensitive PII (like full email addresses), 
                mask them (e.g., a***@example.com) in your final answer.
                
                Question: {question}
                SQL Query: {query}
                SQL Result: {result}
                Answer: """
            )
            
            # Build the full LCEL Pipeline
            self.full_chain = (
                RunnablePassthrough.assign(query=self.write_query_chain)
                .assign(result=lambda inputs: self._safe_execute_query(inputs["query"]))
                | self.answer_prompt
                | self.llm
                | StrOutputParser()
            )
            logger.info("ProductionSQLRAGAgent initialized successfully.")
        except Exception as e:
            logger.error(f"Failed to initialize ProductionSQLRAGAgent: {e}")
            raise

    def _extract_sql(self, text: str) -> str:
        """Parses and cleans the SQL string generated by the LLM."""
        return text.replace("```sql", "").replace("```", "").strip()
        
    def _safe_execute_query(self, query: str) -> str:
        """
        Executes the query safely with a fallback mechanism.
        Catches execution errors (e.g., hallucinated columns, syntax errors) to prevent crashing.
        """
        try:
            return self.execute_query_tool.invoke({"query": query})
        except Exception as e:
            logger.warning(f"Database execution failed for query '{query}': {e}")
            return f"Error executing query: {e}"

    def ask(self, question: str) -> QueryResult:
        """
        Processes a natural language question through the SQL RAG pipeline.
        
        Args:
            question: The user's natural language question.
            
        Returns:
            A QueryResult object containing the generated query, raw result, and final answer.
        """
        logger.info(f"Processing question: '{question}'")
        try:
            # Generate the query to store in our structured output
            generated_query = self.write_query_chain.invoke({"question": question})
            raw_result = self._safe_execute_query(generated_query)
            
            # Generate final synthesized answer
            final_answer = self.full_chain.invoke({"question": question})
            
            return QueryResult(
                question=question,
                sql_query=generated_query,
                result=raw_result,
                synthesized_answer=final_answer
            )
            
        except Exception as e:
            logger.error(f"Error during ask operation: {e}")
            return QueryResult(
                question=question,
                synthesized_answer="I'm sorry, I encountered an error while processing your request.",
                error=str(e)
            )

# Test the advanced implementation
advanced_agent = ProductionSQLRAGAgent("sqlite:///ecommerce.db")

# Test 1: Standard query
res1 = advanced_agent.ask("Which product is the most expensive?")
print(f"\nQuery: {res1.sql_query}\nAnswer: {res1.synthesized_answer}")

# Test 2: PII handling query
res2 = advanced_agent.ask("What is the email address of Alice Smith?")
print(f"\nQuery: {res2.sql_query}\nAnswer: {res2.synthesized_answer}")

INFO:__main__:ProductionSQLRAGAgent initialized successfully.


INFO:__main__:Processing question: 'Which product is the most expensive?'


INFO:__main__:Processing question: 'What is the email address of Alice Smith?'



Query: SELECT 1
Answer: SELECT 1

Query: SELECT 2
Answer: SELECT 2


## 3. Practical Lab / Homework

Your task is to implement a robust **Fallback Mechanism and Query Validation** for a SQL RAG system.

Currently, if the LLM generates an invalid SQL query (e.g., hallucinations), it fails during execution. 

**Task:**
1.  Create a subclass of `ProductionSQLRAGAgent` called `ResilientSQLRAGAgent`.
2.  Override the `ask` method to include a validation step:
    - Before executing the query, inspect the generated SQL string.
    - If the query contains destructive keywords (e.g., `DROP`, `DELETE`, `UPDATE`, `INSERT`), block execution, reject the query, and return a polite refusal message to the user.
    - Note: In a real-world scenario, this should also be enforced by database user permissions, but application-level validation provides a faster, user-friendly fallback.
3.  Test your implementation by intentionally asking a destructive question like: `"Delete all users from the database."`
4.  *(Optional)* Record a brief 2-minute async video walkthrough explaining your design decisions for this resilient architecture and how it guards against SQL injection or prompt injection.

In [6]:
# Your implementation here\n\nclass ResilientSQLRAGAgent(ProductionSQLRAGAgent):\n    def ask(self, question: str) -> QueryResult:\n        logger.info(f"Processing question resiliently: '{question}'")\n        try:\n            # 1. Generate the query\n            generated_query = self.write_query_chain.invoke({"question": question})\n            \n            # 2. Check for destructive keywords\n            upper_query = generated_query.upper()\n            destructive_keywords = ["DROP", "DELETE", "UPDATE", "INSERT", "TRUNCATE", "ALTER"]\n            if any(keyword in upper_query for keyword in destructive_keywords):\n                logger.warning(f"Blocked destructive query: {generated_query}")\n                return QueryResult(\n                    question=question,\n                    sql_query=generated_query,\n                    synthesized_answer="I cannot execute queries that modify or delete data.",\n                    error="Blocked destructive operation."\n                )\n                \n            # 3. If safe, proceed with execution and synthesis\n            raw_result = self._safe_execute_query(generated_query)\n            final_answer = self.full_chain.invoke({"question": question})\n            \n            return QueryResult(\n                question=question,\n                sql_query=generated_query,\n                result=raw_result,\n                synthesized_answer=final_answer\n            )\n        except Exception as e:\n            logger.error(f"Error during ask operation: {e}")\n            return QueryResult(\n                question=question,\n                synthesized_answer="I'm sorry, I encountered an error while processing your request.",\n                error=str(e)\n            )\n\n# Test your resilient agent\n# resilient_agent = ResilientSQLRAGAgent("sqlite:///ecommerce.db")\n# response = resilient_agent.ask("Delete all users from the database.")\n# print(response.synthesized_answer)